# Malaria Detection using Custom CNN

This notebook demonstrates the process of training and evaluating a custom Convolutional Neural Network (CNN) for detecting malaria from microscopic blood cell images.

In [ ]:
import os
import tensorflow as tf
import matplotlib.pyplot as plt
import sys

# Add parent directory to path to import our modules
sys.path.append('..')
from model import build_malaria_cnn
from utils import load_data, plot_training_history, evaluate_model

## 1. Load the Dataset

Ensure the dataset is located in `../data/cell_images/`.

In [ ]:
DATA_DIR = '../data/cell_images'
BATCH_SIZE = 32
IMG_SIZE = (128, 128)

if os.path.exists(DATA_DIR):
    train_ds, val_ds = load_data(DATA_DIR, img_size=IMG_SIZE, batch_size=BATCH_SIZE)
else:
    print(f"Dataset not found at {DATA_DIR}. Please download and extract it.")

## 2. Build the Model

In [ ]:
model = build_malaria_cnn(input_shape=(128, 128, 3))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

model.summary()

## 3. Train the Model

In [ ]:
if os.path.exists(DATA_DIR):
    if not os.path.exists('../saved_model'):
        os.makedirs('../saved_model')
        
    checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
        filepath='../saved_model/malaria_cnn.keras',
        save_best_only=True,
        monitor='val_loss'
    )
    
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True,
        monitor='val_loss'
    )
    
    reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6
    )
    
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15, # Shorter for notebook demonstration
        callbacks=[checkpoint_cb, early_stopping_cb, reduce_lr_cb]
    )
    
    plot_training_history(history, save_dir='../results')

## 4. Evaluate the Model

In [ ]:
if os.path.exists(DATA_DIR):
    evaluate_model(model, val_ds, save_dir='../results')